# Шаг 4

In [11]:
import plotly.graph_objects as go
import networkx as nx
import matplotlib.pyplot as plt
import random
import numpy as np
import copy
import matplotlib.patches as patches
import pandas as pd
import base64
from IPython.display import display, HTML
import ipynbname
from pathlib import Path
from pyvis.network import Network
import json
import random
import matplotlib.cm as cm
import matplotlib.colors as mcolors




def show_html_file(filename):
    try:
        with open(filename, "rb") as f:
            html_content = f.read()
            
        b64_content = base64.b64encode(html_content).decode('utf-8')
        data_url = f'data:text/html;charset=utf-8;base64,{b64_content}'
        iframe = f'<iframe src="{data_url}" width="100%" height="650px" style="border: 1px solid #ddd; border-radius: 5px;"></iframe>'
        
        display(HTML(iframe))
        
    except FileNotFoundError:
        print(f"Файл {filename} не найден. Проверьте путь.")
        

## 4.1. Найти датасет (набор данных) с большими данными для формирования специального графа (иерархического, фрактального, масштабно-инвариантного и др.) с многими весами (3-4 веса ребер, 2-3 веса вершин).

#### Выбор и описание набора данных (Dataset)

Для формирования специального графа был выбран набор данных **OpenFlights: Airport, Airline and Route Database**.

*   **Источник:** [OpenFlights.org Data Page](https://openflights.org/data.html)
*   **Описание:** Набор данных содержит информацию о более чем 10,000 аэропортах, авиакомпаниях и ~70,000 авиамаршрутах по всему миру. Данные представлены в виде нескольких CSV-файлов, что удобно для обработки.
*   **Выбранные файлы для работы:** `airports.dat`, `routes.dat` и `airlines.dat`.

#### Обоснование выбора

Данный набор данных был выбран по следующим причинам:
1.  **Соответствие типу графа:** Сеть авиаперевозок является классическим примером **масштабно-инвариантной (scale-free) и иерархической сети**. В ней присутствуют крупные узлы-хабы (например, аэропорты в Лондоне, Атланте, Дубае) с огромным количеством связей и множество мелких региональных аэропортов с небольшим числом соединений. Такая структура "hub-and-spoke" идеально подходит для анализа фрактальных и иерархических свойств.
2.  **Наличие данных для весов:** Датасет содержит богатую атрибутивную информацию (географические координаты, типы маршрутов, авиакомпании), которая позволяет сформировать несколько осмысленных весов для вершин и рёбер.
3.  **Уникальность:** Данный набор данных не пересекается с датасетами, выбранными другими участниками группы.

---

### Модель графа

Граф будет построен по следующей модели:
*   **Вершины (Nodes):** Аэропорты мира. Каждый аэропорт будет являться уникальным узлом в графе.
*   **Рёбра (Edges):** Прямые авиамаршруты между аэропортами. Ребро существует, если хотя бы одна авиакомпания выполняет рейс между двумя аэропортами.

### Веса Вершин (Аэропорты)

Будут сформированы следующие **3 веса** для каждой вершины:

1.  **`degree` (Количество маршрутов):**
    *   **Тип:** Числовой (целое).
    *   **Значение:** Топологическая характеристика, показывающая связность аэропорта в сети. Рассчитывается напрямую из данных о маршрутах.
2.  **`passengers_per_year` (Годовой пассажиропоток):**
    *   **Тип:** Числовой (целое).
    *   **Значение:** Ключевой показатель "важности" или "мощности" узла. Требует обогащения данных из внешних источников (например, Wikipedia).
3.  **`altitude` (Высота над уровнем моря):**
    *   **Тип:** Числовой (в футах).
    *   **Значение:** Физическая характеристика аэропорта, доступная напрямую из файла `airports.dat`.

### Веса Рёбер (Маршруты)

Будут сформированы следующие **4 веса** для каждого ребра:

1.  **`distance` (Расстояние):**
    *   **Тип:** Числовой (в километрах).
    *   **Значение:** Географическое расстояние между двумя аэропортами, рассчитанное по их координатам.
2.  **`num_airlines` (Число авиакомпаний):**
    *   **Тип:** Числовой (целое).
    *   **Значение:** Показывает уровень конкуренции на маршруте. Рассчитывается путем подсчета уникальных авиакомпаний, обслуживающих данное направление.
3.  **`is_codeshare` (Код-шеринг):**
    *   **Тип:** Бинарный (1 - да, 0 - нет).
    *   **Значение:** Показывает наличие коммерческих соглашений между авиакомпаниями на данном маршруте.
4.  **`num_stops` (Число остановок):**
    *   **Тип:** Числовой (целое).
    *   **Значение:** Характеризует сложность маршрута (в данном датасете для прямых рейсов это значение равно 0).

## 4.2. Моделирование структурно-динамической системы. Формирование структуры графа, взвешивание ребер и вершин. Описание структурно-динамической системы.

In [ ]:
AIRPORTS_FILE = 'airports.dat'
ROUTES_FILE = 'routes.dat'

# Эвристический коэффициент для расчета пассажиропотока
# (Примерное среднее число пассажиров на один маршрут в год)
PASSENGER_HEURISTIC_MULTIPLIER = 50000

def haversine(lon1, lat1, lon2, lat2):
    """
    Вычисляет расстояние в километрах между двумя точками на Земле.
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c  # Радиус Земли в км
    return km


# Заголовки для датафреймов, согласно документации OpenFlights
airport_cols = ['Airport ID', 'Name', 'City', 'Country', 'IATA', 'ICAO', 'Latitude', 'Longitude', 'Altitude', 'Timezone', 'DST', 'Tz database time zone', 'Type', 'Source']
routes_cols = ['Airline', 'Airline ID', 'Source airport', 'Source airport ID', 'Destination airport', 'Destination airport ID', 'Codeshare', 'Stops', 'Equipment']

# Загружаем данные, используя '\N' как маркер для NaN
airports_df = pd.read_csv(AIRPORTS_FILE, header=None, names=airport_cols, na_values='\\N')
routes_df = pd.read_csv(ROUTES_FILE, header=None, names=routes_cols, na_values='\\N')

# Очистка данных
# Удаляем аэропорты без ID, IATA кода или координат
airports_df.dropna(subset=['Airport ID', 'IATA', 'Latitude', 'Longitude'], inplace=True)
# Удаляем маршруты, где неизвестен ID аэропорта
routes_df.dropna(subset=['Source airport ID', 'Destination airport ID'], inplace=True)

# Преобразуем ID в целые числа для удобства
routes_df['Source airport ID'] = routes_df['Source airport ID'].astype(int)
routes_df['Destination airport ID'] = routes_df['Destination airport ID'].astype(int)

print(f"Загружено {len(airports_df)} аэропортов и {len(routes_df)} маршрутов.")


G = nx.Graph()

for index, row in airports_df.iterrows():
    G.add_node(
        row['Airport ID'],
        name=row['Name'],
        city=row['City'],
        country=row['Country'],
        iata=row['IATA'],
        latitude=row['Latitude'],
        longitude=row['Longitude'],
        # Вес 1: Altitude (Высота)
        altitude=row['Altitude']
    )


# Группируем маршруты, чтобы для каждой пары аэропортов было одно ребро
grouped_routes = routes_df.groupby(['Source airport ID', 'Destination airport ID'])

for (source_id, dest_id), group in grouped_routes:
    if G.has_node(source_id) and G.has_node(dest_id):
        # Вес 1: Расстояние (distance)
        source_node = G.nodes[source_id]
        dest_node = G.nodes[dest_id]
        distance_km = haversine(source_node['longitude'], source_node['latitude'], 
                                dest_node['longitude'], dest_node['latitude'])
        
        # Вес 2: Количество авиакомпаний (num_airlines)
        num_airlines = len(group['Airline ID'].unique())
        
        # Вес 3: Является ли код-шеринговым (is_codeshare)
        is_codeshare = 1 if 'Y' in group['Codeshare'].values else 0
        
        # Вес 4: Количество остановок (num_stops) - для прямых рейсов всегда 0
        num_stops = group['Stops'].iloc[0]

        G.add_edge(
            source_id,
            dest_id,
            distance=distance_km,
            num_airlines=num_airlines,
            is_codeshare=is_codeshare,
            num_stops=num_stops
        )


for node_id in G.nodes():
    # Вес 2: Степень узла (degree)
    degree = G.degree(node_id)
    
    # Вес 3: Пассажиропоток (эвристика)
    passenger_flow = degree * PASSENGER_HEURISTIC_MULTIPLIER + np.random.randint(-50000, 50000)
    
    # Добавляем веса как атрибуты узла
    nx.set_node_attributes(G, {node_id: {'degree': degree, 'passengers_per_year': max(0, passenger_flow)}})


print(f"Итоговый граф содержит {G.number_of_nodes()} вершин и {G.number_of_edges()} рёбер.")

# Пример вывода атрибутов для крупного аэропорта (Хитроу, ID=507)
if G.has_node(507):
    print("\nПример атрибутов для вершины (Аэропорт Хитроу, ID=507):")
    print(G.nodes[507])

# Пример вывода атрибутов для ребра (если оно существует)
if G.has_edge(507, 1382): # LHR-JFK
    print("\nПример атрибутов для ребра (Лондон-Хитроу <-> Нью-Йорк-JFK):")
    print(G.edges[507, 1382])

Загружено 6072 аэропортов и 67240 маршрутов.
Итоговый граф содержит 6072 вершин и 18758 рёбер.

Пример атрибутов для вершины (Аэропорт Хитроу, ID=507):
{'name': 'London Heathrow Airport', 'city': 'London', 'country': 'United Kingdom', 'iata': 'LHR', 'latitude': 51.4706, 'longitude': -0.461941, 'altitude': 83, 'degree': 170, 'passengers_per_year': 8464839}

Пример атрибутов для ребра (Лондон-Хитроу <-> Нью-Йорк-JFK):
{'distance': np.float64(347.1675081044457), 'num_airlines': 3, 'is_codeshare': 1, 'num_stops': np.int64(0)}


In [ ]:
# Списки для хранения координат и информации для вершин (аэропортов)
airport_lons = []
airport_lats = []
airport_hover_texts = []

for node_id, data in G.nodes(data=True):
    airport_lons.append(data.get('longitude'))
    airport_lats.append(data.get('latitude'))
    hover_text = f"<b>{data.get('name', 'N/A')}</b><br>" \
                 f"City: {data.get('city', 'N/A')}, {data.get('country', 'N/A')}<br>" \
                 f"IATA: {data.get('iata', 'N/A')}<br>" \
                 f"Routes: {data.get('degree', 0)}"
    airport_hover_texts.append(hover_text)

# Списки для хранения координат рёбер (маршрутов)
edge_lons = []
edge_lats = []

for u, v in G.edges():
    edge_lons.extend([G.nodes[u]['longitude'], G.nodes[v]['longitude'], None])
    edge_lats.extend([G.nodes[u]['latitude'], G.nodes[v]['latitude'], None])


# Слой с рёбрами (маршрутами) - рисуем его первым, чтобы он был на заднем плане
fig = go.Figure(go.Scattergeo(
    lon=edge_lons,
    lat=edge_lats,
    mode='lines',
    line=dict(width=0.7, color='#333333'),
    opacity=0.4,
    hoverinfo='none' # Не показываем информацию при наведении на линию
))

# Слой с вершинами (аэропортами)
fig.add_trace(go.Scattergeo(
    lon=airport_lons,
    lat=airport_lats,
    hoverinfo='text',
    text=airport_hover_texts,
    mode='markers',
    marker=dict(
        size=4,
        color='skyblue',
        line=dict(width=0.5, color='black')
    )
))


fig.update_layout(
    title_text='<b>Глобальная сеть воздушного транспорта (OpenFlights)</b>',
    showlegend=False,
    geo=dict(
        showland=True,
        landcolor='rgb(243, 243, 243)',
        countrycolor='rgb(204, 204, 204)',
        showocean=True,
        oceancolor='rgb(204, 229, 255)',
        showcountries=True,
        projection_type='natural earth'
    ),
    margin={"r":0,"t":40,"l":0,"b":0}
)

print("Отображение интерактивной карты...")
fig.show()

Подготовка данных для визуализации на карте...
Отображение интерактивной карты...


## 4.3. Визуализация структурно-динамической системы в виде графа (как делали на шагах 1-3).


In [ ]:
MIN_DEGREE_THRESHOLD = 60 

print(f"Фильтрация графа: оставляем только узлы со степенью > {MIN_DEGREE_THRESHOLD}...")
nodes_to_keep = [n for n, d in G.degree() if d > MIN_DEGREE_THRESHOLD]
G_sub = G.subgraph(nodes_to_keep)
print(f"Отфильтрованный граф содержит {G_sub.number_of_nodes()} вершин и {G_sub.number_of_edges()} рёбер.")

# Используем cdn_resources='remote', чтобы избежать проблем с путями к библиотекам
net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", notebook=True, cdn_resources='remote')

# Подготовка шкал
degrees = dict(G_sub.degree())
min_degree = min(degrees.values())
max_degree = max(degrees.values())
log_min = np.log(min_degree + 1)
log_max = np.log(max_degree + 1)

for node_id in G_sub.nodes():
    data = G_sub.nodes[node_id]
    degree = degrees[node_id]
    
    norm_degree = (np.log(degree + 1) - log_min) / (log_max - log_min)
    
    # ПРИВЕДЕНИЕ ТИПОВ: используем float() и int() для совместимости с JSON
    size = float(10 + norm_degree * 40)
    color = mcolors.to_hex(cm.plasma(norm_degree))
    
    title = f"<b>{data.get('name', 'N/A')} ({data.get('iata')})</b><br>" \
            f"City: {data.get('city', 'N/A')}<br>" \
            f"Degree: {int(degree)}<br>" \
            f"Passengers (est.): {int(data.get('passengers_per_year', 0))}"
            
    # Преобразуем сам node_id в int, если это numpy.int64
    net.add_node(int(node_id), label=str(data.get('iata')), size=size, color=color, title=title)

print("Добавление рёбер...")
for u, v in G_sub.edges():
    # Преобразуем ID концов ребра в обычные int
    net.add_edge(int(u), int(v), color="rgba(128, 128, 128, 0.3)", width=0.5)

options = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -100,
            "centralGravity": 0.01,
            "springLength": 100,
            "springConstant": 0.1
        },
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based"
    },
    "interaction": {
        "hover": True
    },
    "nodes": {
        "font": {"size": 14, "color": "white"}
    }
}
net.set_options(json.dumps(options))

output_filename = "step4_3_abstract_graph_visualization.html"
try:
    # Используем непосредственную запись, чтобы избежать ошибок шаблонизатора
    net.write_html(output_filename)
    print(f"\nГотово! Файл сохранен: {output_filename}")
except Exception as e:
    print(f"Ошибка при сохранении: {e}")

# Показать в ячейке
show_html_file(output_filename)

Фильтрация графа: оставляем только узлы со степенью > 60...
Отфильтрованный граф содержит 144 вершин и 3124 рёбер.
Добавление стилизованных узлов...
Добавление рёбер...

Готово! Файл сохранен: step4_3_abstract_graph_visualization.html


c:\Projects\FU\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning:

Consider using IPython.display.IFrame instead

